> **Prerequisite — not standalone.** This notebook has no imports or data loading of
> its own. It reuses the kernel state produced by `03_Tensor_PCA.ipynb`
> (`Xflat`, `pca`, `evr`, `X`, `core`, `factors`, `ranks`, `ccy_factor`,
> `ten_factor`, `n_ccy`, `n_ten`, `tenors`, plus `np`, `tl`, `plt`, `PCA`).
>
> **To run:** open and run `03_Tensor_PCA.ipynb` first, then run this notebook in
> the **same kernel session**. Running it on its own will fail with `NameError`.


# Methodology & Formulas — Classic PCA vs Tensor PCA

This section develops the mathematics behind the two decompositions applied above.
We reuse the variables already computed: the flattened matrix `Xflat`, the fitted
`pca` object, the explained-variance ratios `evr`, the centered tensor `X`, and the
Tucker outputs `core`, `ccy_factor`, `ten_factor`.

## 1. Classic PCA: Mathematical Formulation

Let $A \in \mathbb{R}^{n \times p}$ be the data matrix ($n = 200$ scenarios,
$p = I_2 I_3 = 36$ flattened features). Classic PCA proceeds as:

**Centering.** Subtract the feature mean $\bar{a}_j = \frac{1}{n}\sum_i A_{ij}$:

$$
X = A - \bar{A}, \qquad \bar{A} = \mathbf{1}\,\bar{a}^\top .
$$

**Covariance.** Form the (unbiased) sample covariance:

$$
C = \frac{1}{n-1} X^\top X \in \mathbb{R}^{p \times p}.
$$

**Eigendecomposition.** $C$ is symmetric PSD, so

$$
C = W \Lambda W^\top, \qquad \Lambda = \mathrm{diag}(\lambda_1 \ge \dots \ge \lambda_p \ge 0),
$$

where the columns $w_k$ of $W$ are the **principal components** (loadings) and
$\lambda_k$ is the variance captured by PC $k$.

**Explained variance ratio.**

$$
\mathrm{EVR}_k = \frac{\lambda_k}{\sum_{j=1}^{p}\lambda_j}.
$$

Equivalently, via the SVD $X = U \Sigma V^\top$, we have $w_k = v_k$ and
$\lambda_k = \sigma_k^2 / (n-1)$. Below we verify that the covariance eigenvalues
match `pca.explained_variance_`.

In [ ]:
# Verify: eigenvalues of the sample covariance == pca.explained_variance_.
n = Xflat.shape[0]
C = (Xflat.T @ Xflat) / (n - 1)          # (36, 36) covariance matrix

eigvals, eigvecs = np.linalg.eigh(C)     # ascending
eigvals = eigvals[::-1]                  # descending, to match PCA order

print("Top 6 covariance eigenvalues :", np.round(eigvals[:6], 8))
print("pca.explained_variance_ (top6):", np.round(pca.explained_variance_[:6], 8))
print("max |difference|             :", np.max(np.abs(eigvals[:6] - pca.explained_variance_[:6])))

# Explained variance ratio reproduced from eigenvalues.
evr_from_cov = eigvals / eigvals.sum()
print("\nEVR from covariance (top6):", np.round(evr_from_cov[:6], 6))
print("evr (sklearn, top6)       :", np.round(evr[:6], 6))

## 2. Tensor Notation and Mode-n Unfolding

Instead of flattening, we treat each scenario as a matrix and stack them into a
**third-order tensor**

$$
\mathcal{X} \in \mathbb{R}^{I_1 \times I_2 \times I_3},
\qquad (I_1, I_2, I_3) = (\text{scenarios},\ \text{currencies},\ \text{tenors}) = (200, 4, 9).
$$

The **mode-$n$ unfolding** (matricization) $X_{(n)}$ rearranges the tensor into a
matrix whose columns are the mode-$n$ fibers:

$$
X_{(n)} \in \mathbb{R}^{I_n \times \prod_{m \ne n} I_m}.
$$

So:

- $X_{(1)}$ has shape $200 \times 36$ — rows index **scenarios**,
- $X_{(2)}$ has shape $4 \times 1800$ — rows index **currencies**,
- $X_{(3)}$ has shape $9 \times 800$ — rows index **tenors**.

Note $X_{(1)}$ is (up to column ordering) exactly the flattened matrix `Xflat`.

In [ ]:
# Mode-n unfoldings of the centered tensor X (shape 200 x 4 x 9).
for mode, name in enumerate(["scenario", "currency", "tenor"]):
    Xn = tl.unfold(tl.tensor(X), mode)
    print(f"mode-{mode} ({name:>8}) unfolding X_({mode}): shape {tuple(Xn.shape)}")

# Confirm mode-0 unfolding matches the flattened matrix used by classic PCA.
X0 = tl.unfold(tl.tensor(X), 0)
print("\nmode-0 unfolding == Xflat (same values)?",
      np.allclose(np.asarray(X0), Xflat))

## 3. Tucker Decomposition and HOSVD

The **Tucker model** approximates $\mathcal{X}$ by a small **core tensor**
$\mathcal{G} \in \mathbb{R}^{R_1 \times R_2 \times R_3}$ transformed by a factor
matrix $U^{(n)} \in \mathbb{R}^{I_n \times R_n}$ along each mode:

$$
\mathcal{X} \;\approx\; \mathcal{G}\times_1 U^{(1)} \times_2 U^{(2)} \times_3 U^{(3)}
\;=\; \sum_{r_1=1}^{R_1}\sum_{r_2=1}^{R_2}\sum_{r_3=1}^{R_3}
      g_{r_1 r_2 r_3}\; u^{(1)}_{r_1}\circ u^{(2)}_{r_2}\circ u^{(3)}_{r_3}.
$$

Here $\times_n$ is the **$n$-mode product**: multiplying a tensor by a matrix
along mode $n$ acts on the unfolding as ordinary matrix multiplication,

$$
\mathcal{Y} = \mathcal{G}\times_n U^{(n)} \quad\Longleftrightarrow\quad
Y_{(n)} = U^{(n)} G_{(n)} .
$$

**HOSVD initialization.** The Higher-Order SVD sets each factor matrix to the
leading left singular vectors of the corresponding unfolding:

$$
X_{(n)} = U^{(n)} \Sigma^{(n)} \big(V^{(n)}\big)^\top,
\qquad U^{(n)} \leftarrow U^{(n)}[:, 1{:}R_n],
$$

and then the core is $\mathcal{G} = \mathcal{X}\times_1 U^{(1)\top}
\times_2 U^{(2)\top}\times_3 U^{(3)\top}$. Tucker's ALS refines this from the
HOSVD start (as used with `init="svd"` above).

## 4. Computing Mode-n Factors via SVD

We now compute the HOSVD factor matrices directly with `np.linalg.svd` on each
unfolding and compare the leading singular subspaces to the `ccy_factor` and
`ten_factor` returned by `tucker`. Since factor matrices are only defined up to
sign/rotation within their subspace, we measure agreement by the **subspace
alignment**: the principal angles between $\mathrm{span}(U_\text{svd})$ and
$\mathrm{span}(U_\text{tucker})$. Perfect alignment gives singular values of
$U_\text{svd}^\top U_\text{tucker}$ all equal to 1.

In [ ]:
def subspace_alignment(A, B):
    """Cosines of principal angles between column spaces of A and B."""
    Qa, _ = np.linalg.qr(A)
    Qb, _ = np.linalg.qr(B)
    s = np.linalg.svd(Qa.T @ Qb, compute_uv=False)
    return s

# Mode-2 (currency) HOSVD factor: left singular vectors of X_(2).
X2 = np.asarray(tl.unfold(tl.tensor(X), 1))         # (4, 1800)
U2, S2, _ = np.linalg.svd(X2, full_matrices=False)
U_ccy_svd = U2[:, :ccy_factor.shape[1]]             # (4, 2)

# Mode-3 (tenor) HOSVD factor: left singular vectors of X_(3).
X3 = np.asarray(tl.unfold(tl.tensor(X), 2))         # (9, 800)
U3, S3, _ = np.linalg.svd(X3, full_matrices=False)
U_ten_svd = U3[:, :ten_factor.shape[1]]             # (9, 3)

align_ccy = subspace_alignment(U_ccy_svd, np.asarray(ccy_factor))
align_ten = subspace_alignment(U_ten_svd, np.asarray(ten_factor))

print("Currency-mode subspace alignment (cos of principal angles):", np.round(align_ccy, 6))
print("Tenor-mode    subspace alignment (cos of principal angles):", np.round(align_ten, 6))
print("\n(All ≈ 1.0 ⇒ SVD factors and Tucker factors span the same subspace.)")

## 5. Reconstruction and Parameter Count Comparison

**Relative reconstruction error.** For any approximation $\hat{\mathcal{X}}$,

$$
\varepsilon = \frac{\|\mathcal{X} - \hat{\mathcal{X}}\|_F}{\|\mathcal{X}\|_F},
\qquad \|\mathcal{X}\|_F = \sqrt{\sum_{i,j,k} x_{ijk}^2}.
$$

**Parameter counts (loadings only, excluding per-scenario scores):**

- **Classic PCA**, $r$ components: each loading vector lives in
  $\mathbb{R}^{I_2 I_3}$, giving

$$
P_\text{PCA}(r) = r\,I_2 I_3 = 36\,r.
$$

- **Tucker** with ranks $(R_1, R_2, R_3)$: the currency + tenor factor matrices
  plus the core (the scenario factor $U^{(1)}$ plays the role of scores):

$$
P_\text{Tucker} = I_2 R_2 + I_3 R_3 + R_1 R_2 R_3
= 4 R_2 + 9 R_3 + R_1 R_2 R_3.
$$

The table below tabulates these against the errors already computed.

In [ ]:
# Frobenius norm reference.
X_fro = np.linalg.norm(X)

print(f"{'method':>20} {'rel_err':>12} {'loading_params':>15}")
print("-" * 50)

# Classic PCA sweep.
for r in [1, 2, 3, 4, 6]:
    p = PCA(n_components=r).fit(Xflat)
    Xr = p.inverse_transform(p.transform(Xflat))
    err = np.linalg.norm(Xflat - Xr) / np.linalg.norm(Xflat)
    params = r * n_ccy * n_ten                      # r * I2 * I3
    print(f"{'classic PCA r=' + str(r):>20} {err:>12.4e} {params:>15}")

# Tucker.
Xhat = tl.tucker_to_tensor((core, factors))
tuck_err = np.linalg.norm(X - Xhat) / X_fro
R1, R2, R3 = ranks
tuck_params = n_ccy * R2 + n_ten * R3 + R1 * R2 * R3
print("-" * 50)
print(f"{'Tucker ' + str(ranks):>20} {tuck_err:>12.4e} {tuck_params:>15}")

## 6. Interpreting Mode Factors as Curve Shapes

For a smooth term structure, the tenor-mode covariance is well approximated by a
low-rank operator whose leading eigenvectors are the classic **Nelson–Siegel-like
shapes**:

- **Level** ($U^{(3)}_{\cdot,1}$): roughly constant across tenors — a parallel
  shift of the whole curve.
- **Slope** ($U^{(3)}_{\cdot,2}$): monotone sign change from short to long end —
  steepening / flattening.
- **Curvature** ($U^{(3)}_{\cdot,3}$): "smile" shape, opposite sign in the belly —
  a butterfly move.

Mathematically these arise because the tenor correlation matrix is close to a
Toeplitz/kernel matrix $\rho(\tau_i,\tau_j)=e^{-|\tau_i-\tau_j|/\ell}$, whose
Karhunen–Loève eigenfunctions are increasingly oscillatory (0, 1, 2 sign changes).

Similarly, the currency-mode factors decompose into:

- **Common move** ($U^{(2)}_{\cdot,1}$): same-sign loadings on all currencies — a
  global rates shock.
- **Divergence / basis** ($U^{(2)}_{\cdot,2}$): domestic vs. foreign sign split —
  the cross-currency basis widening/tightening.

The plot below reconstructs the level/slope/curvature basis from `ten_factor`.

In [ ]:
# Plot the tenor-mode factors as level / slope / curvature basis shapes.
labels = ["level", "slope", "curvature"]
colors = ["tab:blue", "tab:orange", "tab:green"]

fig, ax = plt.subplots(figsize=(8, 4.5))
tf = np.asarray(ten_factor)
for j in range(tf.shape[1]):
    v = tf[:, j]
    # Sign-normalize so the level factor is positive for readability.
    if v[np.argmax(np.abs(v))] < 0:
        v = -v
    lbl = labels[j] if j < len(labels) else f"factor {j+1}"
    ax.plot(range(n_ten), v, marker="o", color=colors[j % len(colors)], label=lbl)

ax.axhline(0, color="k", lw=0.6)
ax.set_xticks(range(n_ten)); ax.set_xticklabels(tenors, rotation=90, fontsize=8)
ax.set_xlabel("tenor"); ax.set_ylabel("loading")
ax.set_title("Tenor-mode factors ≈ level / slope / curvature")

# Annotate interpretation directly on the curves.
ax.annotate("≈ constant → parallel shift", xy=(n_ten - 1, tf[-1, 0]),
            xytext=(n_ten - 4, 0.6 * np.max(np.abs(tf))), fontsize=8,
            arrowprops=dict(arrowstyle="->", color="tab:blue"))
ax.annotate("sign change → steepen/flatten", xy=(0, tf[0, 1]),
            xytext=(1, -0.6 * np.max(np.abs(tf))), fontsize=8,
            arrowprops=dict(arrowstyle="->", color="tab:orange"))

ax.legend(fontsize=9)
fig.tight_layout()
plt.show()